# Laboratorio 09.1 — Agente mínimo desde cero

Implementaremos el ciclo observar, decidir, ejecutar y continuar con funciones Python explícitas antes de usar un framework de agentes.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Describir el estado que atraviesa un ciclo de agente.
- Validar una decisión estructurada de herramienta con Pydantic.
- Ejecutar una herramienta permitida y agregar su resultado como observación.
- Limitar pasos y llamadas para evitar ciclos o costos inesperados.

## 2. Conceptos

Un agente mínimo mantiene una solicitud y observaciones, decide una acción, ejecuta una herramienta y vuelve a observar el resultado. La salida del modelo no debe ejecutar código directamente. Los límites de pasos, catálogo cerrado, validación de argumentos y una condición de finalización son controles esenciales.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Solicitud] --> B[Observar estado]
    B --> C[Decidir: herramienta o final]
    C -->|herramienta| D[Validar Pydantic]
    D --> E[Ejecutar función permitida]
    E --> F[Agregar observación]
    F --> B
    C -->|final o límite| G[Respuesta y traza]
```

## 4. Preparación del entorno

El ciclo se implementa visiblemente en el notebook; reutilizamos solo esquemas y funciones de herramientas. La decisión se simula para que la secuencia sea reproducible y no requiere un LLM.

In [ ]:
import json

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.tools import execute_tool
from llm_engineering.agents import AgentDecision

raiz_curso = preparar_entorno()

## 5. Definición del problema

El agente recibe una solicitud de cálculo, delega una vez a la calculadora y termina con una respuesta. Las decisiones JSON simuladas son el dataset de este ejemplo; observamos el control del flujo, no la capacidad de un modelo.

## 6. Línea base

Un programa directo calcula la expresión sin agente. Esta línea base es más simple y tiene una llamada fija; el bucle solo aporta valor cuando debe elegir y encadenar herramientas según observaciones.

In [ ]:
solicitud = 'Calcula 8 * 7 y explica el resultado.'
resultado_directo = execute_tool('calculadora', {'expression': '8 * 7'})
print(resultado_directo.output)

## 7. Implementación

El bucle mantiene `observaciones`, consulta un decisor por paso, valida cada JSON con Pydantic y ejecuta solo herramientas del dispatcher. La secuencia simulada pide calcular, recibe el resultado y luego produce una respuesta final.

In [ ]:
decisiones_simuladas = [
    {'action': 'tool', 'tool_name': 'calculadora', 'arguments': {'expression': '8 * 7'}},
    {'action': 'final', 'final_answer': '8 por 7 es 56.'},
]
observaciones = []
traza = []
respuesta_final = None
max_pasos = 4
for paso in range(max_pasos):
    decision = AgentDecision.model_validate(decisiones_simuladas[paso])
    if decision.action == 'final':
        respuesta_final = decision.final_answer
        traza.append({'paso': paso + 1, 'accion': 'final'})
        break

    resultado = execute_tool(decision.tool_name or '', decision.arguments)
    observacion = {
        'tool_name': resultado.tool_name,
        'success': resultado.success,
        'output': resultado.output,
    }
    observaciones.append(observacion)
    traza.append({'paso': paso + 1, 'accion': 'tool', **observacion})

print('Observaciones:', observaciones)
print('Respuesta final:', respuesta_final)
display(pd.DataFrame(traza))

## 8. Experimentos

Variamos el límite de pasos y reutilizamos las mismas decisiones. Si el límite termina antes de recibir `final`, el agente informa agotamiento en vez de asumir éxito.

In [ ]:
filas_limites = []
for limite in (1, 2, 4):
    pasos_consumidos = min(limite, len(decisiones_simuladas))
    completo = pasos_consumidos == 2 and decisiones_simuladas[1]['action'] == 'final'
    filas_limites.append({
        'max_pasos': limite,
        'pasos_consumidos': pasos_consumidos,
        'completo': completo,
        'llamadas_herramienta': min(1, pasos_consumidos),
    })
tabla_limites = pd.DataFrame(filas_limites)
display(tabla_limites)
tabla_limites.to_csv(raiz_curso / 'outputs' / '09_01_agent_step_limits.csv', index=False)

## 9. Evaluación

Evaluamos condición de finalización, número de pasos y llamadas de herramienta. El límite evita un bucle ilimitado, pero un sistema de producción también necesita timeout, presupuesto, deduplicación de llamadas y registro de errores.

In [ ]:
metricas_agente = pd.DataFrame([{
    'completado': respuesta_final is not None,
    'pasos': len(traza),
    'llamadas_herramienta': sum(paso['accion'] == 'tool' for paso in traza),
    'llamadas_llm': 0,
    'respuesta_no_vacia': bool(respuesta_final),
}])
display(metricas_agente)
metricas_agente.to_csv(raiz_curso / 'outputs' / '09_01_agent_metrics.csv', index=False)

## 10. Discusión

El bucle es simple y auditable, pero su decisor simulado no razona sobre solicitudes arbitrarias. El crecimiento del historial puede consumir contexto; las herramientas pueden fallar y una observación no debe convertirse automáticamente en una instrucción confiable.

## 11. Ejercicios

1. Añade una segunda herramienta y decide cuál usar según la solicitud.
2. Simula un error de ejecución y pide al decisor que elija entre reintentar o finalizar.
3. Añade un presupuesto máximo de llamadas separado del número de pasos.
4. Registra el tamaño del historial en cada ciclo.

## 12. Desafío

Sustituye el decisor simulado por un LLM local con salida Pydantic y ejecuta un benchmark corto. Limita el máximo a tres llamadas de modelo y dos de herramienta; registra todas las decisiones y los fallos.

## 13. Ideas clave

- Un agente es un ciclo de control alrededor de herramientas.
- Las decisiones se validan antes de ejecutarse.
- Observaciones alimentan el siguiente paso y hacen crecer el contexto.
- Límites de pasos y llamadas son salvaguardas necesarias.